# 06 - Effect Size Analysis

Statistical significance testing and effect size interpretation for dissertation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import math

plt.rcParams.update({"font.family": "serif", "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})

# Effect size functions
def cohens_d(a, b):
    n1, n2 = len(a), len(b)
    var1, var2 = a.var(ddof=1), b.var(ddof=1)
    pooled_std = math.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    return (a.mean() - b.mean()) / pooled_std if pooled_std > 0 else 0

def cliffs_delta(a, b):
    n1, n2 = len(a), len(b)
    greater = sum(1 for x in a for y in b if x > y)
    less = sum(1 for x in a for y in b if x < y)
    return (greater - less) / (n1 * n2)

def interpret_d(d):
    d = abs(d)
    if d < 0.2: return "negligible"
    elif d < 0.5: return "small"
    elif d < 0.8: return "medium"
    else: return "large"

try:
    %store -r df
except:
    np.random.seed(42)
    n = 20000
    df = pd.DataFrame({
        "algorithm": np.repeat(["RSA-2048", "Kyber-768"], n // 2),
        "latency_us": np.concatenate([
            np.random.lognormal(7.3, 0.4, n // 2),  # RSA
            np.random.lognormal(4.4, 0.3, n // 2),  # Kyber
        ]).astype(int),
    })


In [ ]:
# Pairwise effect size analysis
if "algorithm" in df.columns:
    algorithms = sorted(df["algorithm"].unique())
    
    print("Pairwise Effect Size Analysis")
    print("=" * 60)
    
    results = []
    for i, algo_a in enumerate(algorithms):
        for algo_b in algorithms[i+1:]:
            a = df[df["algorithm"] == algo_a]["latency_us"].values
            b = df[df["algorithm"] == algo_b]["latency_us"].values
            
            d = cohens_d(a, b)
            delta = cliffs_delta(a, b)
            ks_stat, ks_p = stats.ks_2samp(a, b)
            
            print(f"\n{algo_a} vs {algo_b}:")
            print(f"  Cohen's d: {d:.3f} ({interpret_d(d)})")
            print(f"  Cliff's δ: {delta:.3f}")
            print(f"  KS test: statistic={ks_stat:.3f}, p={ks_p:.2e}")
            
            results.append({
                "Comparison": f"{algo_a} vs {algo_b}",
                "Cohen's d": d,
                "Interpretation": interpret_d(d),
                "Cliff's δ": delta,
                "KS statistic": ks_stat,
            })
    
    results_df = pd.DataFrame(results)
    print("\n" + "=" * 60)
    print("\nSummary Table:")
    print(results_df.to_string(index=False))


# 06 - Effect Size Analysis

Compute statistical significance and effect sizes for dissertation.

## Objectives
- Compute Cohen's d, Hedges' g, Glass's delta
- Calculate Cliff's Delta (non-parametric)
- Run Kolmogorov-Smirnov tests
- Compute Wasserstein distance
- Interpret effect sizes for academic writing


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

def cohens_d(a, b):
    """Cohen's d: standardized mean difference."""
    n1, n2 = len(a), len(b)
    var1, var2 = a.var(ddof=1), b.var(ddof=1)
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    return (a.mean() - b.mean()) / pooled_std if pooled_std > 0 else 0

def cliffs_delta(a, b):
    """Cliff's Delta: non-parametric effect size."""
    greater = sum(1 for x in a for y in b if x > y)
    less = sum(1 for x in a for y in b if x < y)
    n = len(a) * len(b)
    return (greater - less) / n if n > 0 else 0

# Load two experiments to compare
EXP_A = "../data/exp_kyber/merged/merged.parquet"
EXP_B = "../data/exp_rsa/merged/merged.parquet"

try:
    df_a = pd.read_parquet(EXP_A)
    df_b = pd.read_parquet(EXP_B)
    print(f"Experiment A: {len(df_a):,} records")
    print(f"Experiment B: {len(df_b):,} records")
except FileNotFoundError as e:
    print(f"File not found: {e}")


In [ ]:
# Compute effect sizes
lat_a = df_a["latency_us"].values
lat_b = df_b["latency_us"].values

d = cohens_d(lat_a, lat_b)
cliff = cliffs_delta(lat_a[:1000], lat_b[:1000])  # Sample for speed
ks_stat, ks_pval = stats.ks_2samp(lat_a, lat_b)
wasserstein = stats.wasserstein_distance(lat_a, lat_b)

print("Effect Size Analysis")
print("=" * 50)
print(f"Cohen's d:          {d:.4f}")
print(f"  Interpretation:   {'negligible' if abs(d) < 0.2 else 'small' if abs(d) < 0.5 else 'medium' if abs(d) < 0.8 else 'large'}")
print(f"\nCliff's Delta:      {cliff:.4f}")
print(f"  Interpretation:   {'negligible' if abs(cliff) < 0.147 else 'small' if abs(cliff) < 0.33 else 'medium' if abs(cliff) < 0.474 else 'large'}")
print(f"\nK-S Statistic:      {ks_stat:.4f}")
print(f"K-S p-value:        {ks_pval:.2e}")
print(f"  Significant:      {'Yes' if ks_pval < 0.05 else 'No'}")
print(f"\nWasserstein Dist:   {wasserstein:.2f} μs")
